# Compare Training Experiments

This notebook compares training runs across different approaches:
- Baseline PF-CNN (trained on TorchSig)
- PF-CNN + MDA-DMC augmentation
- CLSR-AMC (Contrastive Learning with Self-Reconstruction)

Compares validation curves, final accuracy, and robustness metrics.

In [ ]:
import sys
from pathlib import Path

src_path = Path("../src")
if src_path.exists():
    sys.path.insert(0, str(src_path.resolve()))

import matplotlib.pyplot as plt
import torch

## 1. Load Training Histories

In [ ]:
CHECKPOINTS_DIR = Path("../checkpoints")

# Expected checkpoint paths
checkpoint_paths = {
    "Baseline": CHECKPOINTS_DIR / "pfcnn_torchsig" / "best_model.pt",
    "MDA-DMC": CHECKPOINTS_DIR / "pfcnn_augmented" / "best_model.pt",
    "CLSR-AMC": CHECKPOINTS_DIR / "clsr_amc" / "best_model.pt",
}

def load_history(path):
    """Load training history from checkpoint."""
    if not path.exists():
        return None
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    return ckpt.get("history", {})

histories = {}
for name, path in checkpoint_paths.items():
    history = load_history(path)
    if history:
        histories[name] = history
        n_epochs = len(history.get("train_loss", []))
        print(f"Loaded: {name} ({n_epochs} epochs)")
    else:
        print(f"Not found: {name} at {path}")

if not histories:
    print("\nNo checkpoints found! Run training scripts first:")
    print("  uv run python scripts/train_pfcnn.py")
    print("  uv run python scripts/train_pfcnn.py --augment")

## 2. Validation Accuracy Comparison

In [ ]:
if histories:
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ['#2563eb', '#dc2626', '#22c55e']
    
    for (name, history), color in zip(histories.items(), colors):
        if "val_acc" in history:
            epochs = range(1, len(history["val_acc"]) + 1)
            ax.plot(epochs, history["val_acc"], '-', linewidth=2, 
                   label=name, color=color)
    
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Validation Accuracy")
    ax.set_title("Validation Accuracy Comparison")
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()
else:
    print("No histories to plot.")

## 3. Training Loss Comparison

In [ ]:
if histories:
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ['#2563eb', '#dc2626', '#22c55e']
    
    for (name, history), color in zip(histories.items(), colors):
        if "train_loss" in history:
            epochs = range(1, len(history["train_loss"]) + 1)
            ax.plot(epochs, history["train_loss"], '-', linewidth=2,
                   label=name, color=color)
    
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Training Loss")
    ax.set_title("Training Loss Comparison")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 4. Comprehensive Diagnostic

In [ ]:
if histories:
    n_models = len(histories)
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    colors = ['#2563eb', '#dc2626', '#22c55e']
    
    # Val accuracy
    for (name, history), color in zip(histories.items(), colors):
        if "val_acc" in history:
            epochs = range(1, len(history["val_acc"]) + 1)
            axes[0, 0].plot(epochs, history["val_acc"], '-', linewidth=2, label=name, color=color)
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].set_ylabel("Accuracy")
    axes[0, 0].set_title("Validation Accuracy")
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Train loss
    for (name, history), color in zip(histories.items(), colors):
        if "train_loss" in history:
            epochs = range(1, len(history["train_loss"]) + 1)
            axes[0, 1].plot(epochs, history["train_loss"], '-', linewidth=2, label=name, color=color)
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].set_ylabel("Loss")
    axes[0, 1].set_title("Training Loss")
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Val loss
    for (name, history), color in zip(histories.items(), colors):
        if "val_loss" in history:
            epochs = range(1, len(history["val_loss"]) + 1)
            axes[1, 0].plot(epochs, history["val_loss"], '-', linewidth=2, label=name, color=color)
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].set_ylabel("Loss")
    axes[1, 0].set_title("Validation Loss")
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Learning rate (if available)
    has_lr = False
    for (name, history), color in zip(histories.items(), colors):
        if "lr" in history:
            epochs = range(1, len(history["lr"]) + 1)
            axes[1, 1].plot(epochs, history["lr"], '-', linewidth=2, label=name, color=color)
            has_lr = True
    if has_lr:
        axes[1, 1].set_xlabel("Epoch")
        axes[1, 1].set_ylabel("Learning Rate")
        axes[1, 1].set_title("Learning Rate Schedule")
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
        axes[1, 1].set_yscale('log')
    else:
        axes[1, 1].text(0.5, 0.5, 'LR not recorded', ha='center', va='center', fontsize=12)
        axes[1, 1].set_title("Learning Rate")
    
    plt.tight_layout()
    plt.show()

## 5. CLSR-AMC Loss Breakdown (if available)

In [ ]:
if "CLSR-AMC" in histories:
    clsr_history = histories["CLSR-AMC"]
    
    # Check for component losses
    components = ["train_contrastive", "train_classification", "train_reconstruction"]
    available = [c for c in components if c in clsr_history]
    
    if available:
        fig, ax = plt.subplots(figsize=(10, 6))
        colors = ['#2563eb', '#dc2626', '#22c55e']
        
        for comp, color in zip(available, colors):
            epochs = range(1, len(clsr_history[comp]) + 1)
            label = comp.replace("train_", "").title()
            ax.plot(epochs, clsr_history[comp], '-', linewidth=2, label=label, color=color)
        
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.set_title("CLSR-AMC Loss Component Breakdown")
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    else:
        print("CLSR-AMC checkpoint doesn't have loss component breakdown")
else:
    print("CLSR-AMC model not found.")

## 6. Summary Table

In [ ]:
if histories:
    print(f"{'Model':<15} {'Best Val Acc':>12} {'Final Val Acc':>14} {'Epochs':>8}")
    print("-" * 52)
    
    for name, history in histories.items():
        val_acc = history.get("val_acc", [])
        if val_acc:
            best_acc = max(val_acc)
            final_acc = val_acc[-1]
            n_epochs = len(val_acc)
            print(f"{name:<15} {best_acc:>12.4f} {final_acc:>14.4f} {n_epochs:>8}")
        else:
            print(f"{name:<15} {'N/A':>12} {'N/A':>14} {'N/A':>8}")
    
    print("\nInterpretation:")
    print("  - Best Val Acc: Peak validation accuracy during training")
    print("  - Final Val Acc: Accuracy at last epoch")
    print("  - Gap indicates overfitting if Best >> Final")

## 7. Training Diagnostics

In [ ]:
if histories:
    print("Training Diagnostics:")
    print("=" * 50)
    
    for name, history in histories.items():
        print(f"\n{name}:")
        
        val_acc = history.get("val_acc", [])
        train_loss = history.get("train_loss", [])
        val_loss = history.get("val_loss", [])
        
        if val_acc:
            # Check for plateau
            if len(val_acc) > 10:
                last_10 = val_acc[-10:]
                improvement = max(last_10) - min(last_10)
                if improvement < 0.01:
                    print(f"  - Potential plateau: last 10 epochs show < 1% improvement")
            
            # Check for overfitting
            if train_loss and val_loss and len(val_loss) > 5:
                train_trend = train_loss[-1] - train_loss[-5]
                val_trend = val_loss[-1] - val_loss[-5]
                if train_trend < 0 and val_trend > 0:
                    print(f"  - Potential overfitting: train loss decreasing, val loss increasing")
            
            # Best epoch
            best_epoch = val_acc.index(max(val_acc)) + 1
            print(f"  - Best epoch: {best_epoch} (accuracy: {max(val_acc):.4f})")
            
            # Early stopping check
            if len(val_acc) > best_epoch + 10:
                print(f"  - Note: {len(val_acc) - best_epoch} epochs after best - consider early stopping")